# Phân tích và Khám phá Dữ liệu (Exploratory Data Analysis)
Mục tiêu của Notebook này là kiểm tra tính toàn vẹn của tập dữ liệu "Animals Detection Images Dataset". 
Chúng ta sẽ tiến hành nạp file cấu hình `data.yaml`, đọc ngẫu nhiên một vài bức ảnh và vẽ đè các hộp giới hạn (Bounding Box) để xác thực xem tọa độ nhãn YOLO có được gán chính xác hay không.

In [ ]:
# Nạp các thư viện cốt lõi
import os
import cv2
import yaml
import random
import matplotlib.pyplot as plt

# Thiết lập đường dẫn cục bộ (do Notebook đang nằm trong thư mục notebooks/, cần lùi lại 1 cấp '..')
YAML_PATH = '../datasets/animal_data/data.yaml'

# Đọc tệp cấu hình để lấy danh sách class
with open(YAML_PATH, 'r') as file:
    data_config = yaml.safe_load(file)

class_names = data_config['names']
print(f"Tổng số lớp đối tượng: {data_config['nc']}")
print(f"Danh sách 5 lớp đầu tiên: { {k: class_names[k] for k in range(5)} }")

In [ ]:
def plot_yolo_image(image_path, label_path, class_names):
    """Hàm đọc ảnh và vẽ Bounding Box từ file txt chuẩn YOLO."""
    # Đọc ma trận ảnh bằng OpenCV và chuyển không gian màu từ BGR sang RGB
    img = cv2.imread(image_path)
    if img is None:
        print(f"Không thể đọc ảnh: {image_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w, _ = img.shape

    # Kiểm tra xem file nhãn có tồn tại không
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                # Tọa độ YOLO được chuẩn hóa (0-1) dạng: x_center, y_center, width, height
                x_center, y_center, bbox_w, bbox_h = map(float, parts[1:5])

                # Giải mã ngược về tọa độ pixel tuyệt đối của OpenCV
                xmin = int((x_center - bbox_w / 2) * img_w)
                ymin = int((y_center - bbox_h / 2) * img_h)
                xmax = int((x_center + bbox_w / 2) * img_w)
                ymax = int((y_center + bbox_h / 2) * img_h)

                # Vẽ hộp giới hạn (Màu đỏ tươi, độ dày 2)
                cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)
                
                # Gắn nhãn văn bản lên hộp giới hạn
                label_text = class_names.get(class_id, "Unknown")
                cv2.putText(img, label_text, (xmin, ymin - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    else:
        print(f"Ảnh này không có file nhãn: {label_path}")

    # Hiển thị ảnh bằng Matplotlib
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
# Kiểm thử mù: Bốc ngẫu nhiên 1 bức ảnh trong tập huấn luyện để hiển thị
# Lưu ý: Sửa đường dẫn train_images_dir nếu dữ liệu của bạn nằm ở cấu trúc khác
train_images_dir = os.path.join('..', data_config['path'], data_config['train'])
train_labels_dir = train_images_dir.replace('images', 'labels')

# Lấy danh sách toàn bộ ảnh jpg
all_images = [f for f in os.listdir(train_images_dir) if f.endswith('.jpg') or f.endswith('.png')]

if len(all_images) > 0:
    # Bốc ngẫu nhiên 1 file ảnh
    random_img_name = random.choice(all_images)
    
    img_path = os.path.join(train_images_dir, random_img_name)
    # File nhãn YOLO sẽ có cùng tên với file ảnh nhưng đuôi là .txt
    label_path = os.path.join(train_labels_dir, random_img_name.rsplit('.', 1)[0] + '.txt')
    
    print(f"Đang hiển thị ảnh: {random_img_name}")
    plot_yolo_image(img_path, label_path, class_names)
else:
    print("Thư mục trống hoặc chưa giải nén dữ liệu ảnh!")